# 11 — Figures

Regenerates both figures. Neither is redrawn from stored constants: every
plotted value is recomputed from the committed workbooks in this notebook, and
each figure ends with a printed self-check whose numbers must match
`../paper_numbers.yaml`.

| Figure | File | What it shows |
|---|---|---|
| **Figure 2** | `fig_ipi_zones.{png,pdf}` | IPI zone transitions under romanisation, with the two WMT24 Latin-script controls |
| **Figure 1** | `fig_comet_qn.{png,pdf}` | Raw COMET vs COMET-QN across the ten (language, script) cells |

**Inputs:** `../data/indic/indic_parity_xlmr.xlsx`, `../data/latin/wmt24_ende_enes_metrics.xlsx`
**Outputs:** `../results/figures/fig_ipi_zones.{png,pdf}`, `../results/figures/fig_comet_qn.{png,pdf}`

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── Per-language colours and label placement ─────────────────────────────────
COLOURS = {"GUJ": "#2e7d32", "TAM": "#6a1b9a", "MAL": "#1565c0",
           "MAR": "#8d6e63", "HIN": "#d84315"}
LABEL_SIDE = {"GUJ": -1, "TAM": 1, "MAL": 1, "MAR": 1, "HIN": -1}

## Step 1 — Load the Workbooks

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

## Step 2 — Recompute the IPI Values Figure 1 Plots

IPI = |mean IP − 1.0|, native and romanised, for the five Indic languages plus
the two WMT24 controls.

In [ ]:
IPI_NAT = {l: abs(full[l][COL_IP_NAT].mean() - 1.0) for l in LANG_ORDER}
IPI_ROM = {l: abs(full[l][COL_IP_ROM].mean() - 1.0) for l in LANG_ORDER}

latin_ipi = {}
for sheet, iso in [("German", "DEU"), ("Spanish", "SPA")]:
    ip = pd.to_numeric(pd.read_excel(DATA_LATIN, sheet_name=sheet)["target_xlmr_IP"],
                       errors="coerce")
    latin_ipi[iso] = abs(ip.mean() - 1.0)
IPI_DEU, IPI_SPA = latin_ipi["DEU"], latin_ipi["SPA"]

print(f"{'Lang':>5}  {'IPI_nat':>8}  {'IPI_rom':>8}  {'\u0394':>7}")
print("-" * 32)
for l in LANG_ORDER:
    print(f"{l:>5}  {IPI_NAT[l]:>8.3f}  {IPI_ROM[l]:>8.3f}  {IPI_ROM[l] - IPI_NAT[l]:>+7.3f}")
print(f"{'DEU':>5}  {IPI_DEU:>8.3f}")
print(f"{'SPA':>5}  {IPI_SPA:>8.3f}")

assert all(IPI_ROM[l] > 0.70 for l in LANG_ORDER), "all five must land in Paradox"
assert IPI_SPA < 0.05 < IPI_DEU < 0.70
print("\n\u2713 All five Indic languages cross into the Paradox zone (IPI > 0.70)")
print("\u2713 ENG-SPA in Parity, ENG-DEU in Burden")

## Step 3 — Figure 2: IPI Zone Transitions

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.9), dpi=300)

ax.axhspan(0, 0.05, color="#c8e6c9", alpha=0.70, lw=0)      # Parity
ax.axhspan(0.35, 0.70, color="#ffe0b2", alpha=0.60, lw=0)   # Burden
ax.axhspan(0.70, 1.00, color="#ffcdd2", alpha=0.55, lw=0)   # Paradox
ax.axhline(0.70, color="#c62828", ls="--", lw=1)

for i, lang in enumerate(LANG_ORDER):
    c = COLOURS[lang]
    ax.annotate("", xy=(i, IPI_ROM[lang] - 0.012),
                xytext=(i, IPI_NAT[lang] + 0.012),
                arrowprops=dict(arrowstyle="-|>", color=c, lw=1.4))
    ax.plot(i, IPI_NAT[lang], "o", color=c, ms=6, zorder=5)
    ax.plot(i, IPI_ROM[lang], "^", color=c, ms=7, zorder=5)
    off = 0.11 * LABEL_SIDE[lang]
    ax.text(i + off, (IPI_NAT[lang] + IPI_ROM[lang]) / 2,
            f"+{IPI_ROM[lang] - IPI_NAT[lang]:.3f}",
            fontsize=6.0, color=c, va="center",
            ha="left" if LABEL_SIDE[lang] > 0 else "right")

# Latin-script controls, plotted left of the Indic block
ax.plot(-0.62, IPI_DEU, "D", color="#546e7a", ms=6, zorder=5)
ax.text(-0.62, IPI_DEU + 0.08, f"ENG-DEU\n({IPI_DEU:.3f})",
        fontsize=5.4, ha="center", va="bottom", color="#37474f", linespacing=1.15)
ax.plot(-0.62, IPI_SPA, "s", color="#00897b", ms=6, zorder=5)
ax.text(-0.62, IPI_SPA + 0.096, f"ENG-SPA\n({IPI_SPA:.3f})",
        fontsize=5.4, ha="center", va="bottom", color="#00695c", linespacing=1.15)

ax.annotate(f"Malayalam rom.\nIPI = {IPI_ROM['MAL']:.3f}",
            xy=(2.06, IPI_ROM["MAL"] + 0.004), xytext=(2.30, 0.945),
            fontsize=5.7, ha="left", va="center",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#2e7d32", lw=0.8),
            arrowprops=dict(arrowstyle="-", color="#2e7d32", lw=0.8))

ax.text(4.45, 0.885, "Paradox (> 0.70)", fontsize=6, color="#b71c1c",
        ha="right", style="italic")
ax.text(2.00, 0.375, "Burden (0.35\u20130.70)", fontsize=6, color="#a15c00",
        ha="center", style="italic")
ax.text(4.45, 0.022, "Parity (< 0.05)", fontsize=6, color="#1b5e20",
        ha="right", style="italic")

ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", mfc="#424242", ms=5, label="Native script"),
    Line2D([0], [0], marker="^", color="w", mfc="#424242", ms=6, label="Romanised"),
    Line2D([0], [0], marker="D", color="w", mfc="#546e7a", ms=5, label="ENG-DEU (control)"),
    Line2D([0], [0], marker="s", color="w", mfc="#00897b", ms=5, label="ENG-SPA (control)"),
], fontsize=5.3, loc="center right", bbox_to_anchor=(1.0, 0.19), framealpha=0.92)

ax.set_xticks(np.arange(len(LANG_ORDER)))
ax.set_xticklabels(LANG_ORDER, fontsize=7.5)
ax.set_xlim(-1.0, 4.55)
ax.set_ylim(-0.035, 1.02)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.spines["bottom"].set_position(("data", -0.035))
ax.set_ylabel(r"IPI = |IP(L) $-$ 1.0|", fontsize=7.5)
ax.tick_params(axis="y", labelsize=7)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

plt.tight_layout(pad=0.4)
for ext in ("png", "pdf"):
    plt.savefig(FIGURES_DIR / f"fig_ipi_zones.{ext}")
plt.close(fig)

print("[ok] fig_ipi_zones.{png,pdf}")
print(f"     self-check IPI_rom: " +
      "  ".join(f"{l}={IPI_ROM[l]:.3f}" for l in LANG_ORDER))

## Step 4 — Figure 1: COMET-QN Before and After

In [ ]:
def qn_map(scores, reference_sorted):
    frac = stats.rankdata(scores) / (len(scores) + 1)
    return np.quantile(reference_sorted, frac)


REFERENCE = np.sort(np.concatenate([work[l][COL_COMET_NAT].values for l in LANG_ORDER]))

fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.55), dpi=300, sharey=True)


def panel(ax, transform, title):
    position = 0
    for lang in LANG_ORDER:
        for condition, column in (("nat", COL_COMET_NAT), ("rom", COL_COMET_ROM)):
            values = transform(work[lang][column].values)
            bp = ax.boxplot([values], positions=[position], widths=0.62,
                            patch_artist=True, showfliers=False, whis=(5, 95), zorder=3)
            face = COLOURS[lang] if condition == "nat" else "white"
            for box in bp["boxes"]:
                box.set(facecolor=face, edgecolor=COLOURS[lang], lw=0.9,
                        alpha=0.85 if condition == "nat" else 1.0)
            for element in ("whiskers", "caps"):
                for item in bp[element]:
                    item.set(color=COLOURS[lang], lw=0.8)
            for median in bp["medians"]:
                median.set(color="#212121" if condition == "nat" else COLOURS[lang], lw=1.1)
            position += 1
        position += 0.7
    ax.set_title(title, fontsize=8.5, pad=5, fontweight="bold")
    ax.set_xticks([0.5 + i * 2.7 for i in range(len(LANG_ORDER))])
    ax.set_xticklabels(LANG_ORDER, fontsize=7.5)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", lw=0.4, color="#e0e0e0", zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


panel(axes[0], lambda v: v, "Raw COMET")
panel(axes[1], lambda v: qn_map(v, REFERENCE), "After COMET-QN")
axes[0].set_ylabel("COMET score", fontsize=7.5)
axes[0].legend(handles=[
    Line2D([0], [0], marker="s", color="w", mfc="#757575", mec="#424242", ms=6,
           label="Native script"),
    Line2D([0], [0], marker="s", color="w", mfc="white", mec="#424242", ms=6,
           label="Romanised"),
], fontsize=6.4, loc="lower left", framealpha=0.92, handlelength=1.2, borderpad=0.4)

plt.tight_layout(pad=0.5, w_pad=1.2)
for ext in ("png", "pdf"):
    plt.savefig(FIGURES_DIR / f"fig_comet_qn.{ext}")
plt.close(fig)

print("[ok] fig_comet_qn.{png,pdf}")

## Step 5 — Figure 1 Self-Check

The caption quotes the mean absolute per-language gap before and after
renormalisation. It must equal the Table 8 value from notebook 06.

In [ ]:
gaps_before = [abs(work[l][COL_COMET_NAT].mean() - work[l][COL_COMET_ROM].mean())
               for l in LANG_ORDER]
gaps_after = [abs(qn_map(work[l][COL_COMET_NAT].values, REFERENCE).mean()
                  - qn_map(work[l][COL_COMET_ROM].values, REFERENCE).mean())
              for l in LANG_ORDER]
mean_before, mean_after = np.mean(gaps_before), np.mean(gaps_after)

print(f"     mean |per-language gap|: before {mean_before:.2f}, after {mean_after:.2f}")

assert abs(mean_before - 8.14) < 0.01, f"gap before = {mean_before:.2f}, expected 8.14"
assert mean_after < 0.01, f"gap after = {mean_after:.4f}, expected \u2248 0.00"
print(f"\n\u2713 Figure 1 caption values {mean_before:.2f} \u2192 {mean_after:.2f} match "
      f"table7_comet_qn.csv (notebook 06)")

## Step 6 — Output Manifest

In [ ]:
print("=== Notebook 11 — output manifest ===")
for f in sorted(FIGURES_DIR.glob("fig_*")):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1